Импорт необходимых библиотек

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer   
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate

Baseline предобработка 

In [2]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

fare_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_scaled = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch', 'Pclass']),
    ('fare', fare_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

Менее требовательная предобработка для архитектур на деревьях

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

fare_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch', 'Pclass']),
    ('fare', fare_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

Способ оценки как у baseline

In [4]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

Загрузим данные

In [5]:
df = pd.read_csv('E:/ML/titanic-ml/data/raw/train.csv')
y = df['Survived']
X = df.drop(columns='Survived')

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.2,
    shuffle=True,
    random_state=42,
    stratify=y
)

Рассмотрим следующие модели:
* Logistic Regression + L1/L2
* Decision Tree
* kNN
* SVM
* Random Forest
* Gradient Boosting
* XGBoost

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

Зададим модели

In [7]:
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', LogisticRegression())
    ]),

    'kNN': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', KNeighborsClassifier())
    ]),

    'SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC())
    ]),

    'Decision Tree': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', DecisionTreeClassifier())
    ]),

    'Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', RandomForestClassifier())
    ]),

    'Gradient Boosting': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier())
    ]),

    'XGBoost': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', XGBClassifier())
    ])
}

Функция оценки

In [ ]:
def evaluate_models(models, X, y, cv, scoring):
    results = []

    for name, model in models.items():
        cv_results = cross_validate(
            model,
            X,
            y,
            cv=cv,
            scoring=scoring
        )

        result = {'Model': name}

        for metric in scoring:
            scores = cv_results[f'test_{metric}']

            result[f'{metric}_mean'] = scores.mean()
            result[f'{metric}_std'] = scores.std()

        results.append(result)

    return pd.DataFrame(results)

In [10]:
results = evaluate_models(models, X_train, y_train, cv, scoring)

Отсортируем модели по accuracy и ROC AUC

In [18]:
results.sort_values(
    ['accuracy_mean', 'roc_auc_mean'],
    ascending=False
).round(3)

,Model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,Logistic Regression,0.769,0.081,0.714,0.116,0.644,0.188,0.669,0.140,0.829,0.105
2,SVM,0.758,0.048,0.723,0.104,0.633,0.157,0.658,0.088,0.813,0.072
5,Gradient Boosting,0.758,0.082,0.702,0.143,0.659,0.109,0.676,0.109,0.804,0.067
3,Decision Tree,0.747,0.055,0.688,0.111,0.660,0.129,0.662,0.078,0.719,0.059
4,Random Forest,0.746,0.075,0.689,0.137,0.629,0.141,0.650,0.112,0.784,0.067
1,kNN,0.741,0.065,0.692,0.124,0.601,0.139,0.635,0.100,0.790,0.073
6,XGBoost,0.735,0.088,0.692,0.189,0.631,0.189,0.636,0.134,0.769,0.071


Из лучших моделей с базовыми настройками выберем 3 и проведем эксперемент с настройками гиперпараметров и предобработкой

Модели для детального анализа: **Logistic Regression, SVM, Gradient Boosting**

In [19]:
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', LogisticRegression())
    ]),

    'SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC())
    ]),

    'Gradient Boosting': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier())
    ]),
}

In [ ]:
results = evaluate_models(models, X_train, y_train, cv, scoring)
results

Рассмотрим **Logistic Regression**

Создадим сетку для Grid Search

In [35]:
param_grid = {

    "classifier__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ],

    "classifier__class_weight": [
        None,
        "balanced"
    ]
}

In [40]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    models['Logistic Regression'],
    param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(
    X_train,
    y_train
)

print(grid.best_params_)
print(f"{grid.best_score_:.3f}")


{'classifier__C': 0.1, 'classifier__class_weight': 'balanced'}
0.774


Рассмотрим **SVM**

In [47]:
param_dist = {

    "classifier__C": np.logspace(-3,3,20),

    "classifier__gamma": [
        "scale",
        "auto"
    ],

    "classifier__kernel":[
        "rbf",
        "linear"
    ]

}

In [49]:
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    models['SVM'],
    param_distributions=param_dist,
    n_iter=30,
    cv=cv,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

search.fit(
    X_train,
    y_train
)

print(search.best_params_)
print(f"{search.best_score_:.3f}")

{'classifier__kernel': 'linear', 'classifier__gamma': 'auto', 'classifier__C': np.float64(0.1623776739188721)}
0.791


Рассмотрим **Gradient Boosting**

In [50]:
gb_params = {

    "classifier__n_estimators": [
        50,
        100,
        200,
        300
    ],

    "classifier__learning_rate": [
        0.01,
        0.05,
        0.1
    ],

    "classifier__max_depth": [
        2,
        3,
        5
    ],

    "classifier__min_samples_split": [
        2,
        5,
        10
    ],

    "classifier__min_samples_leaf": [
        1,
        3,
        5
    ],

    "classifier__subsample": [
        0.8,
        1.0
    ]
}

In [52]:
search_gb = RandomizedSearchCV(
    models['Gradient Boosting'],
    gb_params,
    n_iter=50,
    cv=cv,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)


search_gb.fit(
    X_train,
    y_train
)

print(search_gb.best_params_)
print(f"{search_gb.best_score_:.3f}")

{'classifier__subsample': 1.0, 'classifier__n_estimators': 200, 'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 1, 'classifier__max_depth': 5, 'classifier__learning_rate': 0.01}
0.797


**Gradient Boosting** показал лучший результат **accuracy = 0.797** при следующих параметрах:  
subsample: 1.0  
n_estimators: 200  
min_samples_split: 10    
min_samples_leaf: 1   
max_depth: 5  
learning_rate': 0.01

Лучшая модель:

Рассмотрим, что можно сделать с обработкой данных, чтобы улучшить качество

In [131]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

fare_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch', 'Pclass']),
    ('fare', fare_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier(
            subsample=1,
            n_estimators=200,
            min_samples_split=10,
            min_samples_leaf=1,
            max_depth=5,
            learning_rate=0.01,
            random_state=42
        ))
    ])
}

results_base = evaluate_models(models, X_train, y_train, cv, scoring)
results_base

,Model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,Best model Random Forest,0.802698,0.053073,0.755604,0.064553,0.702198,0.133524,0.724059,0.099466,0.798826,0.071381


Рассмотрим различные подходы к подготовке данных для лучшей модели

Другая стратегия заполнения Age - mean

In [136]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
])

fare_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch', 'Pclass']),
    ('fare', fare_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier(
            subsample=1,
            n_estimators=200,
            min_samples_split=10,
            min_samples_leaf=1,
            max_depth=5,
            learning_rate=0.01,
            random_state=42
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")
print(f"Прирост roc_auc: {results['roc_auc_mean'].values[0] - results_base['roc_auc_mean'].values[0]:.3f}")


Прирост accuracy: -0.034
Прирост roc_auc: -0.012


Ухудшение качества

Уберем логарифмирование параметра Fare

In [140]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch', 'Pclass','Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier(
            subsample=1,
            n_estimators=200,
            min_samples_split=10,
            min_samples_leaf=1,
            max_depth=5,
            learning_rate=0.01,
            random_state=42
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")
print(f"Прирост roc_auc: {results['roc_auc_mean'].values[0] - results_base['roc_auc_mean'].values[0]:.3f}")


Прирост accuracy: 0.000
Прирост roc_auc: 0.000


нет разницы

добвим логарифмирование параметру SibSp

In [141]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

fare_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'Parch', 'Pclass']),
    ('fare', fare_pipeline, ['Fare', 'SibSp']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier(
            subsample=1,
            n_estimators=200,
            min_samples_split=10,
            min_samples_leaf=1,
            max_depth=5,
            learning_rate=0.01,
            random_state=42
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")
print(f"Прирост roc_auc: {results['roc_auc_mean'].values[0] - results_base['roc_auc_mean'].values[0]:.3f}")


Прирост accuracy: 0.000
Прирост roc_auc: 0.001


Есть прирост roc_auc

Добавим логарифмирование Parch

In [143]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

fare_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'Pclass', 'SibSp']),
    ('fare', fare_pipeline, ['Fare', 'Parch']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier(
            subsample=1,
            n_estimators=200,
            min_samples_split=10,
            min_samples_leaf=1,
            max_depth=5,
            learning_rate=0.01,
            random_state=42
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")
print(f"Прирост roc_auc: {results['roc_auc_mean'].values[0] - results_base['roc_auc_mean'].values[0]:.3f}")


Прирост accuracy: 0.000
Прирост roc_auc: 0.001


Есть прирост roc_auc

Обработаем Pclass как категориальный признак

In [144]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

fare_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch']),
    ('fare', fare_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked', 'Pclass'])
])

models = {
    'Best model Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier(
            subsample=1,
            n_estimators=200,
            min_samples_split=10,
            min_samples_leaf=1,
            max_depth=5,
            learning_rate=0.01,
            random_state=42
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")
print(f"Прирост roc_auc: {results['roc_auc_mean'].values[0] - results_base['roc_auc_mean'].values[0]:.3f}")


Прирост accuracy: -0.011
Прирост roc_auc: -0.007


Ухудшение качества

Использование степенного преобразование (Yeo-Johnson) вместо логарифмирования

In [155]:
from sklearn.preprocessing import PowerTransformer

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

fare_pipeline = Pipeline([
    ('power_tr', PowerTransformer(method='yeo-johnson')),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch', 'Pclass']),
    ('fare', fare_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier(
            subsample=1,
            n_estimators=200,
            min_samples_split=10,
            min_samples_leaf=1,
            max_depth=5,
            learning_rate=0.01,
            random_state=42
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")
print(f"Прирост roc_auc: {results['roc_auc_mean'].values[0] - results_base['roc_auc_mean'].values[0]:.3f}")


Прирост accuracy: 0.000
Прирост roc_auc: 0.000


без изменения

добавим PowerTransformer к Parch

In [157]:

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
])

pt_pipeline = Pipeline([
    ('power_tr', PowerTransformer(method='yeo-johnson')),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Pclass']),
    ('log', log_pipeline, ['Fare']),
    ('pt', pt_pipeline, ['Parch']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier(
            subsample=1,
            n_estimators=200,
            min_samples_split=10,
            min_samples_leaf=1,
            max_depth=5,
            learning_rate=0.01,
            random_state=42
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")
print(f"Прирост roc_auc: {results['roc_auc_mean'].values[0] - results_base['roc_auc_mean'].values[0]:.3f}")


Прирост accuracy: -0.011
Прирост roc_auc: 0.001


первичная метрика accuracy падает

аналогично для SibSp

In [158]:

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
])

pt_pipeline = Pipeline([
    ('power_tr', PowerTransformer(method='yeo-johnson')),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'Parch', 'Pclass']),
    ('log', log_pipeline, ['Fare']),
    ('pt', pt_pipeline, ['SibSp']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier(
            subsample=1,
            n_estimators=200,
            min_samples_split=10,
            min_samples_leaf=1,
            max_depth=5,
            learning_rate=0.01,
            random_state=42
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")
print(f"Прирост roc_auc: {results['roc_auc_mean'].values[0] - results_base['roc_auc_mean'].values[0]:.3f}")


Прирост accuracy: 0.000
Прирост roc_auc: 0.001


Прирост roc_auc

Итоговый вариант

In [159]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
])

pt_pipeline = Pipeline([
    ('power_tr', PowerTransformer(method='yeo-johnson')),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'Parch', 'Pclass']),
    ('log', log_pipeline, ['Fare']),
    ('pt', pt_pipeline, ['SibSp']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier(
            subsample=1,
            n_estimators=200,
            min_samples_split=10,
            min_samples_leaf=1,
            max_depth=5,
            learning_rate=0.01,
            random_state=42
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)
results

,Model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,Best model Random Forest,0.802698,0.053073,0.755604,0.064553,0.702198,0.133524,0.724059,0.099466,0.799476,0.072272


Зафиксируем модель

In [160]:
best_model = Pipeline([
    ('preprocessor', preprocessor_tree),
    ('classifier', GradientBoostingClassifier(
        subsample=1,
        n_estimators=200,
        min_samples_split=10,
        min_samples_leaf=1,
        max_depth=5,
        learning_rate=0.01,
        random_state=42
    ))
])